# Simulación de Volatilidad TCO (100 Días)
### Análisis de Sensibilidad: Diésel y Escasez de Choferes

Este notebook demuestra cómo un modelo de TCO estático (1.50 €/km) es vulnerable ante la realidad del mercado. 
Modelizamos los costes desacoplando los componentes **Fijos** (Personal/Amortización) y **Variables** (Diésel/Neumáticos).

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta

# 1. Configuración de Parámetros Base (Situación Ideal)
DIAS = 100
BASE_TCO = 1.50  # €/km
KM_DIARIOS_PROMEDIO = 450 

# Desglose TCO Real (Estimación sectorial)
COSTE_FIJO_KM = 0.90  # Personal, Seguros, Amortización, Mantenimiento preventivo
COSTE_VARIABLE_KM = 0.60 # Diésel (aprox 40% del total)

np.random.seed(42) # Reproducibilidad

# 2. Simulación de Volatilidad
# El diésel sigue un camino aleatorio (Random Walk) con volatilidad diaria del 1.5%
volatilidad_diesel = np.exp(np.cumsum(np.random.normal(0, 0.015, DIAS)))
diesel_evolucion = COSTE_VARIABLE_KM * volatilidad_diesel

# Los costes salariales son más rígidos pero pueden sufrir saltos (Shock de mercado)
# Simulamos un incremento del 10% en el día 50 debido a renegociación de convenios/escasez
personal_evolucion = np.full(DIAS, COSTE_FIJO_KM)
personal_evolucion[50:] *= 1.10 

# 3. Cálculo de TCO Real Diario
tco_real = personal_evolucion + diesel_evolucion

# 4. Generación de DataFrame para Visualización
fechas = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(DIAS)]
df = pd.DataFrame({
    'Fecha': fechas,
    'TCO_Estatico': BASE_TCO,
    'TCO_Real': tco_real,
    'Componente_Personal': personal_evolucion,
    'Componente_Diesel': diesel_evolucion
})

# 5. Visualización Mejorada con Plotly (Áreas Apiladas)
fig = go.Figure()

# Componente Personal (Base acumulada)
fig.add_trace(go.Scatter(
    x=df['Fecha'], y=df['Componente_Personal'],
    name='Coste Personal (Fijo)',
    stackgroup='one', 
    line=dict(width=0), 
    fillcolor='rgba(59, 130, 246, 0.4)'
))

# Componente Diésel (Sobre el personal)
fig.add_trace(go.Scatter(
    x=df['Fecha'], y=df['Componente_Diesel'],
    name='Coste Diésel (Variable)',
    stackgroup='one', 
    line=dict(width=0),
    fillcolor='rgba(255, 153, 51, 0.4)'
))

# Línea de Referencia (Estática)
fig.add_trace(go.Scatter(
    x=df['Fecha'], y=df['TCO_Estatico'],
    name='TCO Teórico (1.50 €/km)',
    mode='lines',
    line=dict(color='black', width=2, dash='dash')
))

# TCO Real Combinado (Línea de contorno)
fig.add_trace(go.Scatter(
    x=df['Fecha'], y=df['TCO_Real'],
    name='TCO Real (Total)',
    mode='lines',
    line=dict(color='#1e3a8a', width=3)
))

fig.update_layout(
    title=dict(
        text='<b>Evolución del TCO Real vs. Estimación Estática</b><br><span style="font-size:12px">Simulación 100 días: Volatilidad Diésel + Salto Salarial (Día 50)</span>',
        x=0.05,
        y=0.95
    ),
    xaxis_title='Horizonte Temporal (Días)',
    yaxis_title='Coste (€/km)',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="top", # Cambiamos anclaje para controlar mejor la distancia
        y=-0.4, # Bajamos más la leyenda
        xanchor="center",
        x=0.5
    ),
    hovermode="x unified",
    margin=dict(t=100, b=200) # Más margen inferior para separar el eje X de la leyenda
)

fig.show()

### Conclusiones de la Simulación

1. **Erosión del Margen**: Como se observa, el TCO real rara vez coincide con el estático de 1.50 €. En picos de diésel, el coste sube un **8-12%**, lo que podría convertir una ruta rentable en una pérdida neta.
2. **El Salto de Personal**: El escalón en el día 50 representa el riesgo de la escasez de choferes. Una vez que el mercado sube los salarios, el TCO establece un nuevo "suelo" más alto de forma permanente.
3. **Análisis para el Tribunal**: Esta gráfica justifica por qué es vital tener un **modelo dinámico**. Permite responder: *"Mi sistema no asume 1.50€ como un dogma, sino como un punto de partida que se ajusta a la realidad del mercado mediante cláusulas de gasoil y revisiones salariales"*.